In [1]:
import numpy as np
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

In [2]:
raw = pd.read_parquet('../data/processed/features.parquet', columns=['loan_id', 'approval_date'])
df  = pd.read_parquet('../data/processed/cleaned_features.parquet')

df  = df.reset_index(drop=True)
raw = raw.reset_index(drop=True)
df  = df.merge(raw[['loan_id', 'approval_date']], on='loan_id', how='left')
df['approval_date'] = pd.to_datetime(df['approval_date'])

print(f'Merged shape: {df.shape}')
print(f'Date range: {df["approval_date"].min()} → {df["approval_date"].max()}')

Merged shape: (619655, 47)
Date range: 2023-01-01 00:00:00 → 2026-03-03 00:00:00


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 619655 entries, 0 to 619654
Data columns (total 47 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   user_id                        619655 non-null  int32         
 1   target                         619655 non-null  int32         
 2   global_cardinal                619655 non-null  float64       
 3   cardinal_group                 619655 non-null  object        
 4   product                        619655 non-null  object        
 5   principal_due                  619655 non-null  float64       
 6   credit_limit                   619655 non-null  float64       
 7   limit_util_ratio               619655 non-null  float64       
 8   tenure                         619655 non-null  int32         
 9   age                            619655 non-null  float64       
 10  state                          619655 non-null  object        
 11  

In [4]:
product_map = {
    'QC_CrediBuilder':           'Standard',
    'QC Installment v2':         'Flex',
    'QC Installment v2 Rehab':   'FlexRehab',
    'QC_CrediBuilder Rehab':     'InstallRehab',
    'Bullet v1.0':               'ShortBullet',
    'Bullet v2.0':               'TopBullet',
    'Jumia_CrediBuilder':        'PartnerInstallment',
    'Jumia_CrediBuilder Rehab':  'PartnerRehab',
    'Jumia Prequalification':    'PartnerPrequal',
    'Finpal':                    'TopLoans',
    'Others':                    'Other',
}
df['product_type'] = df['product'].map(product_map).fillna('Other')
df.drop(columns=['product'], inplace=True)

channel_map = {
    'Google Ads':  'Paid_Search',
    'FB Ads':      'Social',
    'Organic':     'Organic',
    'Unknown':     'Unknown',
    'Referrals':   'Referral',
    'Hippo Ads':   'Ad1',
    'InMobi Ads':  'Ad2',
    'AppNext Ads': 'Ad3',
    'Others':      'Other',
}
df['channel_group'] = df['acquisition_channel'].map(channel_map).fillna('Other')
df.drop(columns=['acquisition_channel'], inplace=True)

# Rename cardinal columns to generic names
df.rename(columns={
    'global_cardinal': 'loan_number',
    'cardinal_group':  'loan_sequence_group',   # kept temporarily, dropped later
}, inplace=True)

print('Anonymisation applied.')

Anonymisation applied.


In [5]:
df.sort_values(['user_id', 'approval_date'], inplace=True)
df.reset_index(drop=True, inplace=True)

# Number of prior loans for this user before this loan (0 = first loan)
df['prior_loan_count'] = df.groupby('user_id').cumcount()

df['prior_defaults'] = (
    df.groupby('user_id')['target']
      .transform(lambda x: x.shift(1).expanding().sum().fillna(0))
)

df['prior_default_rate'] = np.where(
    df['prior_loan_count'] > 0,
    df['prior_defaults'] / df['prior_loan_count'],
    np.nan
)

df['ever_defaulted'] = (df['prior_defaults'] > 0).astype(int)

df['days_since_last_loan'] = (
    df.groupby('user_id')['approval_date']
      .transform(lambda x: x.diff().dt.days.fillna(0))
)

print('Prior behaviour features computed.')

Prior behaviour features computed.


In [6]:
# Binary flags derived from EDA findings
df['is_first_loan']       = (df['loan_number'] == 1).astype(int)
df['is_full_utilisation'] = (df['limit_util_ratio'] == 1.0).astype(int)
df['is_medium_tenure']    = ((df['tenure'] > 30) & (df['tenure'] <= 60)).astype(int)

# Log transforms — normalise right-skewed distributions
df['cardinal_log']  = np.log1p(df['loan_number'])
df['principal_log'] = np.log1p(df['principal_due'])
df['salary_log']    = np.log1p(df['salary'])

# Burden score: repayment pressure = (loan / income) × tenure_months
df['burden_score'] = np.where(
    df['salary'] > 0,
    (df['principal_due'] / df['salary']) * (df['tenure'] / 30),
    0
)

# first-time borrower AND borrowing at exactly their full limit
df['full_util_x_c1'] = df['is_full_utilisation'] * df['is_first_loan']

# Bureau-derived ratios computed from fields available at inference
df['arrears_ratio'] = np.where(
    df['total_bureau_accounts'] > 0,
    df['accounts_in_arrears'] / df['total_bureau_accounts'],
    0
)

print('Static engineered features created.')

Static engineered features created.


In [7]:
TRAIN_CUTOFF = pd.Timestamp('2025-03-01')
VAL_CUTOFF   = pd.Timestamp('2025-06-01')

df['period'] = np.where(
    df['approval_date'] < TRAIN_CUTOFF, 'train',
    np.where(df['approval_date'] < VAL_CUTOFF, 'val', 'test')
)

user_periods = df.groupby('user_id')['period'].nunique()
multi_period_users = set(user_periods[user_periods > 1].index)
print(f'\nUsers spanning multiple periods → forced to train: {len(multi_period_users):,}')

df['split'] = np.where(
    df['user_id'].isin(multi_period_users),
    'train',
    df['period']
)

print()
for s in ['train', 'val', 'test']:
    sub = df[df['split'] == s]
    print(f"{s:6s}: {len(sub):>7,} loans | "
          f"{sub['user_id'].nunique():>6,} users | "
          f"DR = {sub['target'].mean():.2%}")

user_split_count = df.groupby('user_id')['split'].nunique()
assert user_split_count.max() == 1, "LEAKAGE: user appears in multiple splits"
print('\nLeakage check passed.')


Users spanning multiple periods → forced to train: 17,931

train : 581,744 loans | 146,104 users | DR = 19.42%
val   :  10,941 loans |  9,033 users | DR = 56.78%
test  :  26,970 loans | 13,460 users | DR = 35.71%

Leakage check passed.


In [8]:
df_train_temp = df[df['split'] == 'train'].copy()

state_dr = (df_train_temp.groupby('state')['target']
            .mean()
            .reset_index()
            .rename(columns={'target': 'state_dr'}))

def assign_tier(dr):
    if dr < 0.18:   
        return 'low_risk'
    elif dr < 0.22: 
        return 'medium_risk'
    else:           
        return 'high_risk'

state_dr['state_risk_tier'] = state_dr['state_dr'].apply(assign_tier)
state_tier_map = dict(zip(state_dr['state'], state_dr['state_risk_tier']))

df['state_risk_tier']     = df['state'].map(state_tier_map).fillna('medium_risk')
tier_order                = {'low_risk': 0, 'medium_risk': 1, 'high_risk': 2}
df['state_risk_tier_enc'] = df['state_risk_tier'].map(tier_order)

print('\nState risk tier distribution (train):')
print(df[df['split']=='train']['state_risk_tier'].value_counts())


State risk tier distribution (train):
state_risk_tier
low_risk       287146
high_risk      182483
medium_risk    112115
Name: count, dtype: int64


In [9]:
# IV RANKING
df_train = df[df['split'] == 'train'].copy()
TARGET   = 'target'

EXCLUDE = {
    TARGET, 'loan_id', 'user_id', 'approval_date', 'split', 'due_date', 'period',
    'loan_number',          
    'principal_due',        
    'salary',               
    'limit_util_ratio',     
    'loan_to_income_ratio', 
    'loan_sequence_group',  
    'bureau_availability',  
    'bureau_risk_category', 
    'state',                
    'state_risk_tier',      
    'emp_status',           
    'marital_status',       
    'gender',               
    'edu_status',           
    'prior_defaults',       
    'prior_default_rate',   
    'ever_defaulted',       
    'accounts_good_condition'
}

candidates = [c for c in df_train.columns if c not in EXCLUDE]
print(f'\nCandidate features for IV: {len(candidates)}')
print(sorted(candidates))

def compute_iv(data, feature, target_col, n_bins=10, min_bin_samples=50):
    tmp = data[[feature, target_col]].dropna().copy()
    if tmp.empty or tmp[target_col].nunique() < 2:
        return 0.0

    if tmp[feature].dtype == object or tmp[feature].nunique() <= 15:
        tmp['bin'] = tmp[feature].astype(str)
    else:
        try:
            tmp['bin'] = pd.qcut(tmp[feature].astype(float),
                                  q=n_bins, duplicates='drop', labels=False)
        except ValueError:
            tmp['bin'] = pd.cut(tmp[feature].astype(float),
                                 bins=n_bins, duplicates='drop', labels=False)

    grp = tmp.groupby('bin')[target_col].agg(['sum', 'count'])
    grp.columns = ['events', 'total']
    grp['non_events'] = grp['total'] - grp['events']
    grp = grp[grp['total'] >= min_bin_samples]

    if grp.empty:
        return 0.0

    total_events     = grp['events'].sum()
    total_non_events = grp['non_events'].sum()
    eps = 0.5

    grp['de']  = (grp['events']     + eps) / (total_events     + eps * len(grp))
    grp['dn']  = (grp['non_events'] + eps) / (total_non_events + eps * len(grp))
    grp['woe'] = np.log(grp['de'] / grp['dn'])
    grp['iv']  = (grp['de'] - grp['dn']) * grp['woe']

    return float(grp['iv'].sum())

print('\nComputing IV ranking (takes ~30s on 400k rows)...')
iv_results = {feat: compute_iv(df_train, feat, TARGET) for feat in candidates}
iv_series  = pd.Series(iv_results).sort_values(ascending=False).round(4)

print('\nFull IV ranking:')
print(iv_series.to_string())

# IV threshold 
selected   = iv_series[iv_series >= 0.02].index.tolist()
dropped_iv = iv_series[iv_series < 0.02].index.tolist()

print(f'\nKept  (IV ≥ 0.02): {len(selected)}')
print(f'Dropped (IV < 0.02): {dropped_iv}')


Candidate features for IV: 41
['accounts_bad_condition', 'accounts_in_arrears', 'age', 'amount_in_arrears', 'arrears_ratio', 'burden_score', 'bureau_account_rating', 'bureau_annual_dti', 'bureau_annual_dti_missing', 'bureau_dsti', 'bureau_dsti_missing', 'bureau_health_score', 'cardinal_log', 'channel_group', 'count_written_off_accounts', 'credit_history_months', 'credit_history_months_missing', 'credit_limit', 'days_since_last_loan', 'distinct_enquiring_lenders', 'distinct_lender_count', 'edu_status_missing', 'full_util_x_c1', 'has_doubtful_account', 'has_written_off_account', 'is_first_loan', 'is_full_utilisation', 'is_medium_tenure', 'is_thin_file', 'principal_log', 'prior_loan_count', 'product_type', 'prop_bad', 'prop_bad_missing', 'salary_log', 'state_risk_tier_enc', 'tenure', 'total_bureau_accounts', 'total_enquiry_count', 'total_monthly_obligations', 'total_outstanding_debt']

Computing IV ranking (takes ~30s on 400k rows)...

Full IV ranking:
cardinal_log                     0.

In [10]:
# CORRELATION FILTER — greedy, IV-sorted
numeric_selected     = df_train[selected].select_dtypes(include=[np.number]).columns.tolist()
categorical_selected = [f for f in selected if f not in numeric_selected]

print(f'\nNumeric candidates for correlation filter:   {len(numeric_selected)}')
print(f'Categorical candidates (pass-through): {len(categorical_selected)}')

corr_matrix       = df_train[numeric_selected].corr().abs()
iv_sorted_numeric = iv_series[numeric_selected].sort_values(ascending=False)

final_numeric = []
dropped_corr  = []

for feat in iv_sorted_numeric.index:
    if not final_numeric:
        final_numeric.append(feat)
        continue
    max_corr = corr_matrix.loc[feat, final_numeric].max()
    if max_corr <= 0.85:
        final_numeric.append(feat)
    else:
        partner = corr_matrix.loc[feat, final_numeric].idxmax()
        dropped_corr.append((feat, partner, round(max_corr, 3)))

final_features = final_numeric + categorical_selected

print(f'\nDropped due to correlation (r > 0.85):')
for feat, partner, r in dropped_corr:
    print(f'  {feat:40s}  r={r}  with {partner}')

print(f'\nFinal feature count: {len(final_features)}')
print(f'\nFinal numeric features: {final_numeric}')
print(f'Final categorical features: {categorical_selected}')


Numeric candidates for correlation filter:   26
Categorical candidates (pass-through): 2

Dropped due to correlation (r > 0.85):
  full_util_x_c1                            r=0.914  with is_first_loan

Final feature count: 27

Final numeric features: ['cardinal_log', 'prior_loan_count', 'is_first_loan', 'credit_limit', 'days_since_last_loan', 'principal_log', 'is_full_utilisation', 'age', 'tenure', 'total_monthly_obligations', 'is_medium_tenure', 'total_bureau_accounts', 'bureau_dsti', 'distinct_lender_count', 'bureau_health_score', 'arrears_ratio', 'total_outstanding_debt', 'state_risk_tier_enc', 'bureau_annual_dti', 'distinct_enquiring_lenders', 'total_enquiry_count', 'prop_bad', 'burden_score', 'salary_log', 'is_thin_file']
Final categorical features: ['product_type', 'channel_group']


In [11]:
# DEFINE C1 AND C2+ FEATURE SETS
#
# C1 model (is_first_loan == 1):
#   - Excludes prior behaviour features (all 0/NaN for first loans)
#   - Bureau and demographic signals are primary
#
# C2+ model (is_first_loan == 0):
#   - Includes prior_loan_count and days_since_last_loan
#   - prior_default_rate and ever_defaulted excluded (IV≈0, business rule)
#
# Public API constraint: all features must be either directly
# inputted by the user or computable from their inputs server-side.
# No live bureau API calls. Bureau features are user-declared.

# Features that are meaningless for C1 (first loan = no history)
PRIOR_BEHAVIOUR_FEATURES = [
    'prior_loan_count',
    'days_since_last_loan',
]

# C1: remove prior behaviour features (they are 0/NaN by definition)
FEATURES_C1 = [f for f in final_features
               if f not in PRIOR_BEHAVIOUR_FEATURES]

# C2+: include all final features including prior behaviour
FEATURES_C2PLUS = final_features.copy()
for feat in PRIOR_BEHAVIOUR_FEATURES:
    if feat not in FEATURES_C2PLUS and feat in df_train.columns:
        FEATURES_C2PLUS.append(feat)

print(f'\nC1  model features ({len(FEATURES_C1)}): {FEATURES_C1}')
print(f'\nC2+ model features ({len(FEATURES_C2PLUS)}): {FEATURES_C2PLUS}')


C1  model features (25): ['cardinal_log', 'is_first_loan', 'credit_limit', 'principal_log', 'is_full_utilisation', 'age', 'tenure', 'total_monthly_obligations', 'is_medium_tenure', 'total_bureau_accounts', 'bureau_dsti', 'distinct_lender_count', 'bureau_health_score', 'arrears_ratio', 'total_outstanding_debt', 'state_risk_tier_enc', 'bureau_annual_dti', 'distinct_enquiring_lenders', 'total_enquiry_count', 'prop_bad', 'burden_score', 'salary_log', 'is_thin_file', 'product_type', 'channel_group']

C2+ model features (27): ['cardinal_log', 'prior_loan_count', 'is_first_loan', 'credit_limit', 'days_since_last_loan', 'principal_log', 'is_full_utilisation', 'age', 'tenure', 'total_monthly_obligations', 'is_medium_tenure', 'total_bureau_accounts', 'bureau_dsti', 'distinct_lender_count', 'bureau_health_score', 'arrears_ratio', 'total_outstanding_debt', 'state_risk_tier_enc', 'bureau_annual_dti', 'distinct_enquiring_lenders', 'total_enquiry_count', 'prop_bad', 'burden_score', 'salary_log', 'is

In [12]:
# Union of all features needed across both models
ALL_MODEL_FEATURES = list(set(FEATURES_C1 + FEATURES_C2PLUS))
SAVE_COLS          = ALL_MODEL_FEATURES + [TARGET, 'user_id', 'split', 'is_first_loan']

# Deduplicate in case is_first_loan is already in ALL_MODEL_FEATURES
SAVE_COLS = list(dict.fromkeys(SAVE_COLS))

print('\nSaving splits...')
for split_name in ['train', 'val', 'test']:
    subset = (df[df['split'] == split_name][SAVE_COLS]
              .reset_index(drop=True))
    path   = f'../data/processed/{split_name}_features.parquet'
    subset.to_parquet(path, index=False)

    c1_sub  = subset[subset['is_first_loan'] == 1]
    c2_sub  = subset[subset['is_first_loan'] == 0]
    print(f'  {split_name:6s}: {len(subset):>7,} total | '
          f'C1={len(c1_sub):,} (DR={c1_sub[TARGET].mean():.2%}) | '
          f'C2+={len(c2_sub):,} (DR={c2_sub[TARGET].mean():.2%})')

# State tier map: state name to encoded integer (0/1/2)
state_tier_enc_map = {
    state: tier_order[tier]
    for state, tier in state_tier_map.items()
}

# Product type map: generic name to integer for LR/RF encoding
product_type_vals = sorted(df['product_type'].dropna().unique().tolist())
product_type_map  = {p: i for i, p in enumerate(product_type_vals)}

# Channel group map
channel_vals = sorted(df['channel_group'].dropna().unique().tolist())
channel_map_enc = {c: i for i, c in enumerate(channel_vals)}

# Training-set medians for imputing missing inputs at inference
inference_defaults = {
    'age':                       float(df_train['age'].median()),
    'salary_log':                float(df_train['salary_log'].median()),
    'tenure':                    float(df_train['tenure'].median()),
    'credit_limit':              float(df_train['credit_limit'].median()),
    'distinct_lender_count':     0,
    'total_enquiry_count':       0,
    'bureau_dsti':               0.0,
    'bureau_annual_dti':         0.0,
    'total_monthly_obligations': 0.0,
    'total_outstanding_debt':    0.0,
    'bureau_health_score':       0,
    'prop_bad':                  0.0,
    'arrears_ratio':             0.0,
    'days_since_last_loan':      0,
}

# scale_pos_weight for LightGBM (computed from training set)
n_neg = int((df_train[TARGET] == 0).sum())
n_pos = int((df_train[TARGET] == 1).sum())
scale_pos_weight = round(n_neg / n_pos, 4)

artifacts = {
    'state_tier_enc_map':   state_tier_enc_map,
    'state_tier_str_map':   state_tier_map,       
    'product_type_map':     product_type_map,
    'channel_map_enc':      channel_map_enc,
    'inference_defaults':   inference_defaults,
    'scale_pos_weight':     scale_pos_weight,
    'features_c1':          FEATURES_C1,
    'features_c2plus':      FEATURES_C2PLUS,
    'target':               TARGET,
    'train_cutoff':         str(TRAIN_CUTOFF.date()),
    'val_cutoff':           str(VAL_CUTOFF.date()),
    'train_default_rate':   round(float(df_train[TARGET].mean()), 4),
    'train_n_loans':        len(df_train),
    'train_n_users':        df_train['user_id'].nunique(),
}

with open('../models/inference_artifacts.json', 'w') as f:
    json.dump(artifacts, f, indent=2)

print('\nInference artifacts saved → ../models/inference_artifacts.json')


Saving splits...
  train : 581,744 total | C1=102,282 (DR=41.24%) | C2+=479,462 (DR=14.77%)
  val   :  10,941 total | C1=8,857 (DR=58.99%) | C2+=2,084 (DR=47.36%)
  test  :  26,970 total | C1=12,379 (DR=45.42%) | C2+=14,591 (DR=27.47%)

Inference artifacts saved → ../models/inference_artifacts.json


In [13]:
iv_series.to_frame('iv').to_csv('../data/processed/iv_ranking.csv')

pd.DataFrame({
    'feature':   FEATURES_C1,
    'model':     'C1',
    'iv':        [iv_series.get(f, 0.0) for f in FEATURES_C1],
}).to_csv('../data/processed/features_c1.csv', index=False)

pd.DataFrame({
    'feature':   FEATURES_C2PLUS,
    'model':     'C2+',
    'iv':        [iv_series.get(f, 0.0) for f in FEATURES_C2PLUS],
}).to_csv('../data/processed/features_c2plus.csv', index=False)

print(f'scale_pos_weight (LightGBM): {scale_pos_weight}')
print(f'\nFiles saved:')

scale_pos_weight (LightGBM): 4.149

Files saved:
